In [ ]:
from datetime import datetime

import lightgbm as lgb
import mlflow
import mlflow.lightgbm
import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

In [ ]:
df = pd.read_csv("../../data/exp01/train.csv")

In [ ]:
mlflow.set_tracking_uri("http://localhost:5001")
mlflow.lightgbm.autolog(disable=True)
experiment = mlflow.get_experiment_by_name("exp01")
run_name = f"01_base_model_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

with mlflow.start_run(experiment_id=experiment.experiment_id, run_name=run_name) as run:
    # Split data into features (X) and target (y)
    X = df.drop(["defect"], axis=1)
    y = df["defect"]

    # First split into train+val (80%) and test (20%)
    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    # Create dataset for LightGBM
    train_data = lgb.Dataset(X_train, label=y_train)
    val_data = lgb.Dataset(X_val, label=y_val)

    # Set parameters for LightGBM
    params = {
        "objective": "binary",
        "metric": "binary_logloss",
        "boosting_type": "gbdt",
        "num_leaves": 31,
        "learning_rate": 0.05,
        "feature_fraction": 0.9,
    }

    # Train model
    num_round = 100
    lgb_model = lgb.train(
        params, train_data, num_round, valid_sets=[train_data, val_data]
    )

    # Evaluate on validation set
    threshold = 0.5
    val_predictions = (lgb_model.predict(X_val) > threshold).astype(int)
    val_accuracy = accuracy_score(y_val, val_predictions)
    print(f"Validation Set Performance:{val_accuracy:0.4f}")
    mlflow.log_metric("val_accuracy", val_accuracy)


In [ ]:
with mlflow.start_run(run_id=run.info.run_id) as run:
    # Evaluate on test set
    df_test = pd.read_csv("../../data/exp01/test.csv")
    X_test = df_test.drop(["defect"], axis=1)
    y_test = df_test["defect"]

    test_predictions = (lgb_model.predict(X_test) > threshold).astype(int)
    test_accuracy = accuracy_score(y_test, test_predictions)
    print(f"Test Set Performance:{test_accuracy:0.4f}")
    mlflow.log_metric("test_accuracy", test_accuracy)

    mlflow.lightgbm.log_model(
        lgb_model,
        artifact_path="model",
        signature=mlflow.models.infer_signature(X_test, test_predictions),
        input_example=X_test.iloc[:5],
    )